# 14a -- SAM2 Masked Stage 2

Kernel slug: `yahiaakhalafallah/14a-sam2-masked-stage2`.

This experiment reuses the 10a Stage 0/1 checkpoint and reruns only Stage 2 with SAM2 foreground masking enabled before vehicle ReID feature extraction. Internal frame IDs remain the 0-based Stage 1 convention; no MOT submission conversion is performed here.

SAM2 weights mechanism: I checked Kaggle for public `sam2_hiera_base_plus` datasets and found third-party uploads, but this kernel uses `enable_internet=true` and downloads Meta's published `sam2_hiera_base_plus.pt` at runtime. That is simpler than adding a new private dataset and avoids depending on a low-usability third-party dataset source.

In [ ]:
import os, sys, subprocess, shutil, json, time, tarfile, re
from pathlib import Path

# Guard: detect GPU before importing torch. Kaggle's newest PyTorch builds may
# not support P100 sm_60, so pin the cu124 build requested for this experiment.
if shutil.which("nvidia-smi"):
    result = subprocess.run(
        ["nvidia-smi", "--query-gpu=gpu_name,compute_cap", "--format=csv,noheader"],
        capture_output=True,
        text=True,
    )
    if result.returncode == 0 and result.stdout.strip():
        gpu_name, compute_cap = result.stdout.strip().split(",", 1)
        match = re.search(r"(\d+)\.(\d+)", compute_cap)
        if match:
            major, minor = match.groups()
            sm = int(major) * 10 + int(minor)
            if sm < 70:
                print(f"GPU {gpu_name.strip()} sm_{sm}: installing torch 2.4.1+cu124")
                subprocess.check_call([
                    sys.executable, "-m", "pip", "install", "-q",
                    "torch==2.4.1+cu124", "torchvision==0.19.1+cu124",
                    "--index-url", "https://download.pytorch.org/whl/cu124",
                ])

import torch

print(f"Python : {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA   : {torch.cuda.is_available()}")
for index in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(index)
    print(f"  GPU {index}: {torch.cuda.get_device_name(index)} ({props.total_memory / 1024**3:.1f} GB)")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

## 1. Clone Repo And Install Dependencies

In [ ]:
REPO_URL = "https://github.com/MRKDaGods/gp.git"
WORK_DIR = Path("/kaggle/working")
PROJECT = WORK_DIR / "gp"

if not PROJECT.exists():
    print(f"Cloning {REPO_URL} ...")
    subprocess.check_call(["git", "clone", "--depth", "1", "-b", "feature/pretrained-ensemble", REPO_URL, str(PROJECT)])
else:
    print("Repo already present; pulling latest ...")
    subprocess.check_call(["git", "-C", str(PROJECT), "pull", "--ff-only"])

os.chdir(str(PROJECT))
sys.path.insert(0, str(PROJECT))
print(f"Repo ready at {PROJECT}")

In [ ]:
def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

# Keep torch/torchvision pinned above; install project dependencies around it.
pip("--no-deps", "ultralytics")
pip("filterpy", "ftfy", "lapx")
pip("--no-deps", "boxmot==11.0.3")

try:
    import torchreid
    print("torchreid ok")
except ImportError:
    pip("git+https://github.com/KaiyangZhou/deep-person-reid.git")

try:
    import faiss
    print(f"faiss ok ({faiss.__version__})")
except ImportError:
    try:
        pip("faiss-gpu")
    except Exception:
        pip("faiss-cpu")

pip("timm", "motmetrics")
pip("loguru", "omegaconf", "rich", "networkx>=3.1", "click")
pip("hydra-core", "iopath")
pip("--no-deps", "git+https://github.com/facebookresearch/sam2.git")

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"], cwd=str(PROJECT))
print("Dependencies installed")

In [ ]:
FAILED = []
for label, module in [
    ("ultralytics", "ultralytics"),
    ("boxmot", "boxmot"),
    ("torch", "torch"),
    ("torchreid", "torchreid"),
    ("timm", "timm"),
    ("faiss", "faiss"),
    ("motmetrics", "motmetrics"),
    ("cv2", "cv2"),
    ("omegaconf", "omegaconf"),
    ("sam2", "sam2"),
]:
    try:
        __import__(module)
        print(f"  OK {label}")
    except ImportError as exc:
        print(f"  MISSING {label}: {exc}")
        FAILED.append(label)
if FAILED:
    raise RuntimeError(f"Missing modules: {FAILED}")
print("All required modules importable")

## 2. Mount MTMC And SAM2 Weights

In [ ]:
import zipfile

WEIGHTS_INPUT = Path("/kaggle/input/mtmc-weights")
if not WEIGHTS_INPUT.exists():
    candidates = [
        path
        for path in Path("/kaggle/input").iterdir()
        if path.is_dir() and "mtmc" in path.name.lower() and "weight" in path.name.lower()
    ]
    if candidates:
        WEIGHTS_INPUT = candidates[0]
    else:
        raise FileNotFoundError(
            "MTMC weights dataset not found; attach gumfreddy/mtmc-weights in kernel-metadata.json"
        )
print(f"MTMC weights input: {WEIGHTS_INPUT}")

MODELS_DST = PROJECT / "models"
if MODELS_DST.is_symlink():
    MODELS_DST.unlink()
if MODELS_DST.exists():
    shutil.rmtree(MODELS_DST)
print(f"Copying models from {WEIGHTS_INPUT} ...")
shutil.copytree(str(WEIGHTS_INPUT), str(MODELS_DST))

for zip_path in sorted(MODELS_DST.rglob("*.zip")):
    print(f"Extracting {zip_path.relative_to(MODELS_DST)}")
    with zipfile.ZipFile(zip_path) as archive:
        archive.extractall(zip_path.parent)
    zip_path.unlink()

for subdir in ["detection", "reid", "tracker", "sam2"]:
    (MODELS_DST / subdir).mkdir(exist_ok=True)
for path in list(MODELS_DST.glob("*.pt")):
    if "yolo" in path.name.lower():
        shutil.move(str(path), str(MODELS_DST / "detection" / path.name))
    elif "sam2" in path.name.lower():
        shutil.move(str(path), str(MODELS_DST / "sam2" / path.name))
    elif "osnet" in path.name.lower():
        shutil.move(str(path), str(MODELS_DST / "tracker" / path.name))
for path in list(MODELS_DST.glob("*.pth")):
    shutil.move(str(path), str(MODELS_DST / "reid" / path.name))
for path in list(MODELS_DST.glob("*.pkl")):
    shutil.move(str(path), str(MODELS_DST / "reid" / path.name))
for path in list(MODELS_DST.glob("*.json")):
    if path.name != "dataset-metadata.json":
        shutil.move(str(path), str(MODELS_DST / "reid" / path.name))

essential = [
    PROJECT / "models" / "reid" / "transreid_cityflowv2_best.pth",
]
missing = [str(path) for path in essential if not path.exists()]
if missing:
    raise FileNotFoundError(f"Missing essential weights: {missing}")
print("Baseline ReID weights ready")

In [ ]:
import urllib.request

SAM2_DIR = PROJECT / "models" / "sam2"
SAM2_DIR.mkdir(parents=True, exist_ok=True)
SAM2_CKPT = SAM2_DIR / "sam2_hiera_base_plus.pt"
SAM2_URL = "https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_base_plus.pt"

if not SAM2_CKPT.exists() or SAM2_CKPT.stat().st_size < 250_000_000:
    print(f"Downloading SAM2 Hiera base-plus from Meta: {SAM2_URL}")
    urllib.request.urlretrieve(SAM2_URL, str(SAM2_CKPT))
size_mb = SAM2_CKPT.stat().st_size / 1024**2
print(f"SAM2 checkpoint ready: {SAM2_CKPT} ({size_mb:.1f} MB)")

In [ ]:
from pathlib import Path

PATCH_PATH = PROJECT / "src" / "stage2_features" / "foreground_masker.py"
PATCH_TEXT = '"""SAM2-based foreground masking for Stage 2 vehicle crops."""\n\nfrom __future__ import annotations\n\nimport logging\nfrom collections.abc import Sequence\nfrom pathlib import Path\nfrom typing import Any\n\nimport cv2\nimport numpy as np\nimport torch\n\nlogger = logging.getLogger(__name__)\n\n_SAM2_MODEL_SPECS = {\n    "sam2_hiera_tiny": {\n        "hf_model_id": "facebook/sam2-hiera-tiny",\n        "config": "configs/sam2/sam2_hiera_t.yaml",\n        "config_21": "configs/sam2.1/sam2.1_hiera_t.yaml",\n        "checkpoint_names": (\n            "sam2_hiera_tiny.pt",\n            "sam2_hiera_t.pt",\n            "sam2.1_hiera_tiny.pt",\n            "sam2.1_hiera_t.pt",\n        ),\n    },\n    "sam2_hiera_small": {\n        "hf_model_id": "facebook/sam2-hiera-small",\n        "config": "configs/sam2/sam2_hiera_s.yaml",\n        "config_21": "configs/sam2.1/sam2.1_hiera_s.yaml",\n        "checkpoint_names": (\n            "sam2_hiera_small.pt",\n            "sam2_hiera_s.pt",\n            "sam2.1_hiera_small.pt",\n            "sam2.1_hiera_s.pt",\n        ),\n    },\n    "sam2_hiera_base_plus": {\n        "hf_model_id": "facebook/sam2-hiera-base-plus",\n        "config": "configs/sam2/sam2_hiera_b+.yaml",\n        "config_21": "configs/sam2.1/sam2.1_hiera_b+.yaml",\n        "checkpoint_names": (\n            "sam2_hiera_base_plus.pt",\n            "sam2_hiera_b+.pt",\n            "sam2.1_hiera_base_plus.pt",\n            "sam2.1_hiera_b+.pt",\n        ),\n    },\n    "sam2_hiera_large": {\n        "hf_model_id": "facebook/sam2-hiera-large",\n        "config": "configs/sam2/sam2_hiera_l.yaml",\n        "config_21": "configs/sam2.1/sam2.1_hiera_l.yaml",\n        "checkpoint_names": (\n            "sam2_hiera_large.pt",\n            "sam2_hiera_l.pt",\n            "sam2.1_hiera_large.pt",\n            "sam2.1_hiera_l.pt",\n        ),\n    },\n}\n\n\ndef _normalise_model_name(model_name: str) -> str | None:\n    name = str(model_name).lower().replace("\\\\", "/")\n    name = Path(name).name\n    if name.endswith((".pt", ".pth")):\n        name = name.removesuffix(".pth").removesuffix(".pt")\n    name = name.replace("facebook/", "")\n    name = name.replace("sam2.1-", "sam2_").replace("sam2-", "sam2_")\n    name = name.replace("hiera-base-plus", "hiera_base_plus")\n    name = name.replace("hiera-tiny", "hiera_tiny")\n    name = name.replace("hiera-small", "hiera_small")\n    name = name.replace("hiera-large", "hiera_large")\n\n    aliases = {\n        "hiera_tiny": "sam2_hiera_tiny",\n        "tiny": "sam2_hiera_tiny",\n        "t": "sam2_hiera_tiny",\n        "hiera_small": "sam2_hiera_small",\n        "small": "sam2_hiera_small",\n        "s": "sam2_hiera_small",\n        "hiera_base_plus": "sam2_hiera_base_plus",\n        "base_plus": "sam2_hiera_base_plus",\n        "b+": "sam2_hiera_base_plus",\n        "hiera_large": "sam2_hiera_large",\n        "large": "sam2_hiera_large",\n        "l": "sam2_hiera_large",\n    }\n    return name if name in _SAM2_MODEL_SPECS else aliases.get(name)\n\n\ndef _iter_model_name_candidates(model_name: str) -> list[str]:\n    model_key = _normalise_model_name(model_name)\n    candidates = [model_name]\n\n    if model_key is not None:\n        candidates.append(_SAM2_MODEL_SPECS[model_key]["hf_model_id"])\n\n    if model_name.startswith("facebook/sam2.1-"):\n        candidates.append(model_name.replace("facebook/sam2.1-", "facebook/sam2-", 1))\n    elif model_name.startswith("facebook/sam2-"):\n        candidates.append(model_name.replace("facebook/sam2-", "facebook/sam2.1-", 1))\n\n    deduped: list[str] = []\n    for candidate in candidates:\n        if candidate not in deduped:\n            deduped.append(candidate)\n    return deduped\n\n\nclass ForegroundMasker:\n    """Masks background in vehicle crops using SAM2 with a center-point prompt."""\n\n    def __init__(\n        self,\n        model_name: str = "facebook/sam2.1-hiera-tiny",\n        min_crop_size: int = 48,\n        fill_value: str | Sequence[int] = "mean",\n        device: str = "cuda:0",\n    ):\n        self.min_crop_size = int(min_crop_size)\n        self.fill_value_mode = fill_value\n        self.device = device\n        self.predictor = self._load_model(model_name)\n\n    def _load_model(self, model_name: str):\n        try:\n            from sam2.sam2_image_predictor import SAM2ImagePredictor\n        except ImportError as exc:\n            logger.error("sam2 package not installed. Install with: pip install sam-2")\n            raise ImportError("sam2 package not installed") from exc\n\n        local_model = self._load_local_model(model_name, SAM2ImagePredictor)\n        if local_model is not None:\n            return local_model\n\n        last_error: Exception | None = None\n        for candidate in _iter_model_name_candidates(model_name):\n            try:\n                predictor = SAM2ImagePredictor.from_pretrained(candidate, device=self.device)\n                logger.info("SAM2 loaded: %s", candidate)\n                return predictor\n            except Exception as exc:  # pragma: no cover - depends on external package/runtime\n                last_error = exc\n\n        raise RuntimeError(f"Unable to load SAM2 model \'{model_name}\'") from last_error\n\n    def _load_local_model(self, model_name: str, predictor_cls: type):\n        checkpoint_path = self._resolve_local_checkpoint(model_name)\n        if checkpoint_path is None:\n            return None\n\n        model_key = _normalise_model_name(str(checkpoint_path)) or _normalise_model_name(model_name)\n        if model_key is None:\n            raise RuntimeError(f"Unable to infer SAM2 model type from checkpoint \'{checkpoint_path}\'")\n\n        spec = _SAM2_MODEL_SPECS[model_key]\n        config_name = spec["config_21"] if "sam2.1" in checkpoint_path.name else spec["config"]\n\n        try:\n            from sam2.build_sam import build_sam2\n\n            model = build_sam2(\n                config_name,\n                str(checkpoint_path),\n                device=self.device,\n                apply_postprocessing=False,\n            )\n            logger.info(\n                "SAM2 loaded from local checkpoint: model=%s config=%s checkpoint=%s",\n                model_key,\n                config_name,\n                checkpoint_path,\n            )\n            return predictor_cls(model)\n        except Exception as exc:  # pragma: no cover - depends on external package/runtime\n            raise RuntimeError(\n                f"Unable to load SAM2 local checkpoint \'{checkpoint_path}\' with config \'{config_name}\'"\n            ) from exc\n\n    def _resolve_local_checkpoint(self, model_name: str) -> Path | None:\n        explicit_path = Path(model_name).expanduser()\n        if explicit_path.suffix.lower() in {".pt", ".pth"}:\n            if explicit_path.exists():\n                return explicit_path\n            if explicit_path.parent != Path("."):\n                raise FileNotFoundError(f"SAM2 checkpoint does not exist: {explicit_path}")\n\n        model_key = _normalise_model_name(model_name)\n        if model_key is None:\n            return None\n\n        checkpoint_names = _SAM2_MODEL_SPECS[model_key]["checkpoint_names"]\n        search_roots = (\n            Path("models/sam2"),\n            Path("models"),\n            Path("checkpoints/sam2"),\n            Path("checkpoints"),\n            Path("/kaggle/working/gp/models/sam2"),\n        )\n\n        for root in search_roots:\n            for checkpoint_name in checkpoint_names:\n                candidate = root / checkpoint_name\n                if candidate.exists():\n                    return candidate\n\n        return None\n\n    def _resolve_fill_value(self, crop_bgr: np.ndarray) -> np.ndarray:\n        if isinstance(self.fill_value_mode, str):\n            fill_value = self.fill_value_mode.lower()\n            if fill_value == "mean":\n                return np.rint(crop_bgr.mean(axis=(0, 1))).astype(np.uint8)\n            if fill_value in {"zero", "black"}:\n                return np.zeros(3, dtype=np.uint8)\n            raise ValueError(\n                "fill_value must be \'mean\', \'zero\', a numeric scalar, or an RGB sequence of length 3"\n            )\n\n        if isinstance(self.fill_value_mode, (int, float)):\n            value = np.clip(float(self.fill_value_mode), 0.0, 255.0)\n            return np.full(3, value, dtype=np.uint8)\n\n        if isinstance(self.fill_value_mode, Sequence) and len(self.fill_value_mode) == 3:\n            return np.asarray(list(self.fill_value_mode), dtype=np.uint8)\n\n        raise ValueError("fill_value must be \'mean\', \'zero\', a numeric scalar, or an RGB sequence of length 3")\n\n    def mask_crop(self, crop_bgr: np.ndarray) -> np.ndarray:\n        """Apply foreground masking to a single BGR crop."""\n        if crop_bgr is None or crop_bgr.ndim != 3 or crop_bgr.shape[2] != 3:\n            return crop_bgr\n\n        height, width = crop_bgr.shape[:2]\n        if min(height, width) < self.min_crop_size:\n            return crop_bgr\n\n        crop_rgb = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB)\n        center_point = np.array([[width // 2, height // 2]], dtype=np.float32)\n        center_label = np.array([1], dtype=np.int32)\n\n        with torch.inference_mode():\n            self.predictor.set_image(crop_rgb)\n            masks, scores, _ = self.predictor.predict(\n                point_coords=center_point,\n                point_labels=center_label,\n                multimask_output=True,\n            )\n\n        if masks is None or len(masks) == 0:\n            return crop_bgr\n\n        best_idx = int(np.argmax(np.asarray(scores))) if scores is not None else 0\n        binary_mask = np.asarray(masks[best_idx], dtype=bool)\n        if binary_mask.shape != (height, width):\n            binary_mask = cv2.resize(\n                binary_mask.astype(np.uint8),\n                (width, height),\n                interpolation=cv2.INTER_NEAREST,\n            ).astype(bool)\n\n        if not np.any(binary_mask):\n            return crop_bgr\n\n        masked = crop_bgr.copy()\n        masked[~binary_mask] = self._resolve_fill_value(crop_bgr)\n        return masked\n\n    def mask_crops(self, crops: list[Any]) -> list[Any]:\n        """Apply foreground masking to quality-scored crops in place."""\n        for crop in crops:\n            crop.image = self.mask_crop(crop.image)\n        return crops'
PATCH_PATH.write_text(PATCH_TEXT, encoding="utf-8")
print(f"Patched {PATCH_PATH} with local SAM2 checkpoint loader")

## 3. Mount CityFlowV2 Videos

In [ ]:
import re as regex

for mount in ["/tmp", "/kaggle/working"]:
    total, used, free = shutil.disk_usage(mount)
    print(f"{mount:16s} {free / 1024**3:.1f} GB free / {total / 1024**3:.1f} GB total")

candidate_mounts = [
    Path("/kaggle/input/data-aicity-2023-track-2"),
    Path("/kaggle/input/datasets/thanhnguyenle/data-aicity-2023-track-2"),
]
CITYFLOW_INPUT = next((path for path in candidate_mounts if path.exists()), None)
if CITYFLOW_INPUT is None:
    raise FileNotFoundError("CityFlowV2 dataset not found; attach thanhnguyenle/data-aicity-2023-track-2")
print(f"CityFlowV2 input: {CITYFLOW_INPUT}")

TMP_DATA = Path("/tmp/datasets")
TMP_DATA.mkdir(parents=True, exist_ok=True)
DATA_RAW_PARENT = PROJECT / "data" / "raw"
if not DATA_RAW_PARENT.is_symlink():
    if DATA_RAW_PARENT.exists():
        shutil.rmtree(DATA_RAW_PARENT)
    DATA_RAW_PARENT.parent.mkdir(parents=True, exist_ok=True)
    DATA_RAW_PARENT.symlink_to(TMP_DATA)

DATA_RAW = TMP_DATA / "cityflowv2"
DATA_RAW.mkdir(parents=True, exist_ok=True)
for split_dir in sorted(CITYFLOW_INPUT.iterdir()):
    if not split_dir.is_dir() or split_dir.name not in ("train", "validation", "test"):
        continue
    for scene_dir in sorted(split_dir.iterdir()):
        if not scene_dir.is_dir():
            continue
        for cam_dir in sorted(scene_dir.iterdir()):
            if not cam_dir.is_dir():
                continue
            flat_name = f"{scene_dir.name}_{cam_dir.name}"
            flat_dir = DATA_RAW / flat_name
            if not flat_dir.exists():
                flat_dir.symlink_to(cam_dir)

cam_pattern = regex.compile(r"^S\d{2}_c\d{3}$")
cams = sorted(path.name for path in DATA_RAW.iterdir() if path.is_dir() and cam_pattern.match(path.name))
print(f"CityFlowV2 ready: {len(cams)} cameras")
for cam in cams:
    print(f"  {cam}")

## 4. Load 10a Stage 0/1 Checkpoint

In [ ]:
from pathlib import Path
import subprocess

PREV_OWNER_SLUG = "yahiaakhalafallah/mtmc-10a-stages-0-2"
PREV_SLUG = PREV_OWNER_SLUG.split("/", 1)[1]
INPUT_ROOT = Path("/kaggle/input")


def find_input_dir(slug: str, owner_slug: str, hints=()) -> Path:
    direct = INPUT_ROOT / slug
    if direct.exists():
        return direct

    owner, _, kernel = owner_slug.partition("/")
    nested = INPUT_ROOT / "notebooks" / owner / kernel
    if nested.exists():
        return nested

    lowered_slug = slug.lower()
    lowered_hints = tuple(str(hint).lower() for hint in hints)
    direct_children = list(INPUT_ROOT.iterdir()) if INPUT_ROOT.exists() else []
    for path in direct_children:
        if not path.is_dir():
            continue
        name = path.name.lower()
        if lowered_slug in name or all(hint in name for hint in lowered_hints):
            return path

    for path in INPUT_ROOT.rglob("checkpoint.tar.gz") if INPUT_ROOT.exists() else []:
        parent_text = str(path.parent).lower()
        if lowered_slug in parent_text or all(hint in parent_text for hint in lowered_hints):
            return path.parent

    return direct


def find_checkpoint() -> Path:
    prev_input = find_input_dir(PREV_SLUG, PREV_OWNER_SLUG, hints=("10a", "stages", "0", "2"))
    checkpoint_path = prev_input / "checkpoint.tar.gz"
    if checkpoint_path.exists():
        print(f"10a input: {prev_input}")
        return checkpoint_path

    print(f"checkpoint.tar.gz not found at {checkpoint_path}; trying Kaggle API fallback")
    dl_dir = Path("/tmp/kaggle_10a_download")
    dl_dir.mkdir(parents=True, exist_ok=True)
    for candidate in [PREV_OWNER_SLUG, "gumfreddy/mtmc-10a-stages-0-2"]:
        result = subprocess.run(
            [
                "kaggle",
                "kernels",
                "output",
                candidate,
                "--file-pattern",
                r"^checkpoint\.tar\.gz$",
                "-p",
                str(dl_dir),
            ],
            capture_output=True,
            text=True,
        )
        print(result.stdout)
        print(result.stderr)
        checkpoint_path = dl_dir / "checkpoint.tar.gz"
        if checkpoint_path.exists() and checkpoint_path.stat().st_size > 0:
            print(f"10a checkpoint downloaded from {candidate}")
            return checkpoint_path

    searched = [str(path) for path in INPUT_ROOT.rglob("checkpoint.tar.gz")] if INPUT_ROOT.exists() else []
    raise FileNotFoundError(
        f"10a checkpoint.tar.gz not found for {PREV_OWNER_SLUG}; searched mount candidates and API fallback. "
        f"Visible checkpoints under /kaggle/input: {searched[:20]}"
    )


checkpoint = find_checkpoint()

EXTRACT_DIR = Path("/tmp/10a_checkpoint")
if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Extracting {checkpoint} ({checkpoint.stat().st_size / 1024**2:.1f} MB)")
with tarfile.open(str(checkpoint), "r:gz") as tar:
    tar.extractall(str(EXTRACT_DIR))

with open(EXTRACT_DIR / "run_metadata.json") as handle:
    previous_meta = json.load(handle)
PREV_RUN_NAME = previous_meta["run_name"]
PREV_RUN_DIR = EXTRACT_DIR / PREV_RUN_NAME
print(f"Loaded 10a run: {PREV_RUN_NAME}")
for stage in ["stage1", "stage2"]:
    stage_dir = PREV_RUN_DIR / stage
    print(f"  {stage}: exists={stage_dir.exists()} files={len([p for p in stage_dir.rglob('*') if p.is_file()]) if stage_dir.exists() else 0}")

## 5. Run Masked Stage 2 Only

In [ ]:
from datetime import datetime

DATA_OUT = Path("/tmp/pipeline_outputs")
DATA_OUT.mkdir(parents=True, exist_ok=True)
RUN_NAME = f"run_14a_sam2_masked_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
RUN_DIR = DATA_OUT / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
shutil.copytree(PREV_RUN_DIR / "stage1", RUN_DIR / "stage1")
print(f"Run  : {RUN_NAME}")
print(f"Stage1 copied from {PREV_RUN_DIR / 'stage1'}")

In [ ]:
os.chdir(str(PROJECT))

from src.core.config import load_config, save_config
from src.core.io_utils import load_tracklets_by_camera
from src.core.logging_utils import setup_logging
from src.stage2_features import run_stage2

BASELINE_WEIGHTS = "models/reid/transreid_cityflowv2_best.pth"
TERTIARY_SLUG = "09s-dinov2-large-cityflowv2"
TERTIARY_OWNER_SLUG = f"yahiaakhalafallah/{TERTIARY_SLUG}"
TERTIARY_FILENAME = "vehicle_transreid_dinov2_large_cityflowv2_final.pth"


def find_tertiary_weights() -> str:
    candidates = [
        Path("/kaggle/input") / TERTIARY_SLUG / TERTIARY_FILENAME,
        Path("/kaggle/input/notebooks/yahiaakhalafallah") / TERTIARY_SLUG / TERTIARY_FILENAME,
    ]
    for candidate in candidates:
        if candidate.exists():
            return str(candidate)

    input_root = Path("/kaggle/input")
    if input_root.exists():
        for candidate in input_root.rglob(TERTIARY_FILENAME):
            if TERTIARY_SLUG in str(candidate) or "dinov2" in str(candidate).lower():
                return str(candidate)

    dl_dir = Path("/tmp/kaggle_09s_download")
    dl_dir.mkdir(parents=True, exist_ok=True)
    result = subprocess.run(
        [
            "kaggle",
            "kernels",
            "output",
            TERTIARY_OWNER_SLUG,
            "--file-pattern",
            f"^{TERTIARY_FILENAME}$",
            "-p",
            str(dl_dir),
        ],
        capture_output=True,
        text=True,
    )
    print(result.stdout)
    print(result.stderr)
    downloaded = dl_dir / TERTIARY_FILENAME
    if downloaded.exists() and downloaded.stat().st_size > 0:
        return str(downloaded)

    return str(candidates[0])


TERTIARY_WEIGHTS = find_tertiary_weights()
print(f"Primary TransReID: {BASELINE_WEIGHTS} exists={Path(BASELINE_WEIGHTS).exists()}")
print(f"DINOv2 tertiary : {TERTIARY_WEIGHTS} exists={Path(TERTIARY_WEIGHTS).exists()}")

overrides = [
    f"project.run_name={RUN_NAME}",
    f"project.output_dir={DATA_OUT}",
    f"stage0.input_dir={DATA_RAW}",
    "stage0.cameras=[S01_c001,S01_c002,S01_c003,S02_c006,S02_c007,S02_c008]",
    "stage2.crop.samples_per_tracklet=48",
    "stage2.crop.foreground_masking.enabled=true",
    "stage2.crop.foreground_masking.model_name=sam2_hiera_base_plus",
    "stage2.crop.foreground_masking.min_crop_size=48",
    "stage2.crop.foreground_masking.fill_value=0",
    "stage2.crop.foreground_masking.dilate_px=5",
    "stage2.crop.foreground_masking.bbox_expand=1.10",
    "stage2.crop.foreground_masking.batch_size=8",
    "stage2.pca.n_components=384",
    "stage2.reid.color_augment=false",
    "stage2.reid.vehicle.concat_patch=false",
    "stage2.reid.vehicle.input_size=[256,256]",
    "stage2.reid.vehicle2.enabled=false",
    "stage2.multi_query.k=0",
    "stage2.camera_bn.enabled=false",
    "stage2.camera_tta.enabled=false",
    "stage2.power_norm.alpha=0.0",
]

if Path(TERTIARY_WEIGHTS).exists():
    overrides += [
        "stage2.reid.vehicle3.enabled=true",
        f"stage2.reid.vehicle3.weights_path={TERTIARY_WEIGHTS}",
        "stage2.reid.vehicle3.input_size=[252,252]",
        "stage2.reid.vehicle3.embedding_dim=1024",
        "stage2.reid.vehicle3.model_name=transreid",
        "stage2.reid.vehicle3.vit_model=vit_large_patch14_dinov2.lvd142m",
        "stage2.reid.vehicle3.clip_normalization=false",
        "stage2.reid.vehicle3.num_cameras=59",
        "stage2.reid.vehicle3.save_separate=true",
    ]
else:
    print("WARNING: DINOv2 tertiary checkpoint not found; vehicle3 will stay disabled")

cfg = load_config("configs/default.yaml", dataset_config="configs/datasets/cityflowv2.yaml", overrides=overrides)
setup_logging(level=cfg.project.get("log_level", "INFO"), log_file=RUN_DIR / "pipeline.log")
save_config(cfg, RUN_DIR / "config.yaml")
video_paths = {}
for cam_dir in sorted(DATA_RAW.glob("S*_c*")):
    if not cam_dir.is_dir():
        continue
    cam_id = cam_dir.name
    video_path = cam_dir / "vdo.avi"
    if video_path.exists():
        video_paths[cam_id] = str(video_path)
tracklets_by_camera = load_tracklets_by_camera(RUN_DIR / "stage1")
print(f"Tracklets: {sum(len(v) for v in tracklets_by_camera.values())} across {len(tracklets_by_camera)} cameras")
print(f"Videos   : {len(video_paths)} discovered")
for camera_id in sorted(tracklets_by_camera):
    print(f"  {camera_id}: tracklets={len(tracklets_by_camera[camera_id])} video={camera_id in video_paths}")

start = time.time()
features = run_stage2(
    cfg,
    tracklets_by_camera,
    video_paths,
    output_dir=RUN_DIR / "stage2",
    smoke_test=False,
    stage0_dir=None,
)
elapsed = time.time() - start
print(f"Masked Stage 2 complete: {len(features)} features in {elapsed / 60:.1f} min")

## 6. Run Stages 3-5

In [ ]:
from src.stage3_indexing import run_stage3
from src.stage4_association import run_stage4
from src.stage5_evaluation import run_stage5

AQE_K = 3
SIM_THRESH = 0.50
SOLVER = "cc"
ALGORITHM = "conflict_free_cc"
LOUVAIN_RES = 0.70

APPEARANCE_WEIGHT = 0.70
HSV_WEIGHT = 0.0
ST_WEIGHT = round(1.0 - APPEARANCE_WEIGHT - HSV_WEIGHT, 4)

BRIDGE_PRUNE = 0.0
MAX_COMP_SIZE = 12
FIC_REG = 0.50
GALLERY_THRESH = 0.48
ORPHAN_MATCH_THRESH = 0.38

INTRA_MERGE = True
INTRA_MERGE_THRESH = 0.80
INTRA_MERGE_GAP = 30

W_TERTIARY = 0.60
W_PRIMARY = round(1.0 - W_TERTIARY, 6)

MULTI_QUERY_WEIGHT = 0.00
CAMERA_BIAS = False
CAMERA_BIAS_ITERS = 2
ZONE_MODEL = False
ZONE_BONUS = 0.06
ZONE_PENALTY = 0.04
HIERARCHICAL = False
HIER_CENTROID_TH = 0.45
HIER_MERGE_TH = 0.45
HIER_ORPHAN_TH = 0.40
RERANKING = False
RERANKING_K1 = 20
RERANKING_K2 = 6
RERANKING_LAMBDA = 0.5
CAMERA_PAIR_NORM = False
AFLINK_ENABLED = False
AFLINK_MAX_TIME_GAP_FRAMES = 150
AFLINK_MAX_SPATIAL_GAP_PX = 150.0
AFLINK_MIN_DIRECTION_COS = 0.85
AFLINK_MIN_VELOCITY_RATIO = 0.5
AFLINK_VELOCITY_WINDOW = 5
MTMC_ONLY = False

PRIMARY_EMBEDDINGS_PATH = RUN_DIR / "stage2" / "embeddings.npy"
TERTIARY_DINOV2_PATH = RUN_DIR / "stage2" / "embeddings_tertiary.npy"
assert PRIMARY_EMBEDDINGS_PATH.exists(), PRIMARY_EMBEDDINGS_PATH
assert TERTIARY_DINOV2_PATH.exists(), (
    f"Missing DINOv2 tertiary embeddings at {TERTIARY_DINOV2_PATH}. "
    "This run must evaluate the production TransReID+DINOv2 score-fusion control."
)

GT_DIR = DATA_RAW
if not any((GT_DIR / cam / "gt" / "gt.txt").exists() for cam in ["S01_c001", "S01_c002", "S01_c003", "S02_c006", "S02_c007", "S02_c008"]):
    dataset_gt = Path("/kaggle/input/data-aicity-2023-track-2")
    extracted_gt = EXTRACT_DIR / "gt_annotations"
    if dataset_gt.exists():
        GT_DIR = dataset_gt
    elif extracted_gt.exists():
        GT_DIR = extracted_gt
    else:
        raise FileNotFoundError("CityFlowV2 ground-truth annotations not found")

stage345_overrides = [
    f"project.run_name={RUN_NAME}",
    f"project.output_dir={DATA_OUT}",
    f"stage0.input_dir={DATA_RAW}",
    "stage0.cameras=[S01_c001,S01_c002,S01_c003,S02_c006,S02_c007,S02_c008]",
    f"stage4.association.query_expansion.k={AQE_K}",
    "stage4.association.query_expansion.alpha=5.0",
    "stage4.association.query_expansion.dba=false",
    f"stage4.association.graph.similarity_threshold={SIM_THRESH}",
    f"stage4.association.solver={SOLVER}",
    f"stage4.association.graph.algorithm={ALGORITHM}",
    f"stage4.association.graph.louvain_resolution={LOUVAIN_RES}",
    f"stage4.association.graph.bridge_prune_margin={BRIDGE_PRUNE}",
    f"stage4.association.graph.max_component_size={MAX_COMP_SIZE}",
    f"stage4.association.weights.vehicle.appearance={APPEARANCE_WEIGHT}",
    f"stage4.association.weights.vehicle.hsv={HSV_WEIGHT}",
    f"stage4.association.weights.vehicle.spatiotemporal={ST_WEIGHT}",
    "stage4.association.mutual_nn.top_k_per_query=20",
    "stage4.association.fic.enabled=true",
    f"stage4.association.fic.regularisation={FIC_REG}",
    f"stage4.association.reranking.enabled={str(RERANKING).lower()}",
    f"stage4.association.reranking.k1={RERANKING_K1}",
    f"stage4.association.reranking.k2={RERANKING_K2}",
    f"stage4.association.reranking.lambda_value={RERANKING_LAMBDA}",
    f"stage4.association.camera_pair_norm.enabled={str(CAMERA_PAIR_NORM).lower()}",
    "stage4.association.fac.enabled=false",
    f"stage4.association.multi_query.enabled={str(MULTI_QUERY_WEIGHT > 0.0).lower()}",
    f"stage4.association.multi_query.weight={MULTI_QUERY_WEIGHT}",
    f"stage4.association.aflink.enabled={str(AFLINK_ENABLED).lower()}",
    f"stage4.association.aflink.max_time_gap_frames={AFLINK_MAX_TIME_GAP_FRAMES}",
    f"stage4.association.aflink.max_spatial_gap_px={AFLINK_MAX_SPATIAL_GAP_PX}",
    f"stage4.association.aflink.min_direction_cos={AFLINK_MIN_DIRECTION_COS}",
    f"stage4.association.aflink.min_velocity_ratio={AFLINK_MIN_VELOCITY_RATIO}",
    f"stage4.association.aflink.velocity_window={AFLINK_VELOCITY_WINDOW}",
    "stage4.association.secondary_embeddings.path=",
    "stage4.association.secondary_embeddings.weight=0.0",
    f"stage4.association.tertiary_embeddings.path={TERTIARY_DINOV2_PATH}",
    f"stage4.association.tertiary_embeddings.weight={W_TERTIARY}",
    f"stage4.association.camera_bias.enabled={str(CAMERA_BIAS).lower()}",
    f"stage4.association.camera_bias.iterations={CAMERA_BIAS_ITERS}",
    f"stage4.association.zone_model.enabled={str(ZONE_MODEL).lower()}",
    "stage4.association.zone_model.zone_data_path=configs/datasets/cityflowv2_zones.json",
    f"stage4.association.zone_model.bonus={ZONE_BONUS}",
    f"stage4.association.zone_model.penalty={ZONE_PENALTY}",
    f"stage4.association.hierarchical.enabled={str(HIERARCHICAL).lower()}",
    f"stage4.association.hierarchical.centroid_threshold={HIER_CENTROID_TH}",
    f"stage4.association.hierarchical.merge_threshold={HIER_MERGE_TH}",
    f"stage4.association.hierarchical.orphan_threshold={HIER_ORPHAN_TH}",
    "stage4.association.hierarchical.max_merge_size=12",
    f"stage4.association.intra_camera_merge.enabled={str(INTRA_MERGE).lower()}",
    f"stage4.association.intra_camera_merge.threshold={INTRA_MERGE_THRESH}",
    f"stage4.association.intra_camera_merge.max_time_gap={INTRA_MERGE_GAP}",
    "stage4.association.gallery_expansion.enabled=true",
    f"stage4.association.gallery_expansion.threshold={GALLERY_THRESH}",
    f"stage4.association.gallery_expansion.orphan_match_threshold={ORPHAN_MATCH_THRESH}",
    "stage4.association.weights.length_weight_power=0.3",
    "stage4.association.temporal_overlap.enabled=true",
    "stage4.association.temporal_overlap.bonus=0.05",
    "stage4.association.temporal_overlap.max_mean_time=5.0",
    f"stage5.mtmc_only_submission={str(MTMC_ONLY).lower()}",
    "stage5.stationary_filter.enabled=true",
    "stage5.stationary_filter.min_displacement_px=150",
    "stage5.stationary_filter.max_mean_velocity_px=2.0",
    "stage5.min_submission_confidence=0.15",
    "stage5.cross_id_nms_iou=0.40",
    "stage5.min_trajectory_confidence=0.30",
    "stage5.min_trajectory_frames=40",
    "stage5.track_edge_trim.enabled=false",
    "stage5.track_smoothing.enabled=false",
    "stage5.gt_frame_clip=true",
    "stage5.gt_zone_filter=true",
    f"stage5.ground_truth_dir={GT_DIR}",
]

cfg = load_config("configs/default.yaml", dataset_config="configs/datasets/cityflowv2.yaml", overrides=stage345_overrides)
save_config(cfg, RUN_DIR / "config.yaml")

print("Locked downstream config:")
print(f"  Stage 3: FAISS {cfg.stage3.faiss.index_type}")
print(f"  Stage 4: AQE_K={AQE_K}, sim_thresh={SIM_THRESH}, algorithm={ALGORITHM}")
print(f"  weights: primary={W_PRIMARY:.2f}, tertiary_dinov2={W_TERTIARY:.2f}, secondary=0.00")
print(f"  FIC_REG={FIC_REG}, gallery={GALLERY_THRESH}, orphan={ORPHAN_MATCH_THRESH}")
print(f"  Stage 5 GT: {GT_DIR}")

start = time.time()
faiss_index, metadata_store = run_stage3(
    cfg,
    features,
    tracklets_by_camera,
    output_dir=RUN_DIR / "stage3",
)
stage3_min = (time.time() - start) / 60.0
print(f"Stage 3 complete in {stage3_min:.2f} min")

start = time.time()
trajectories = run_stage4(
    cfg,
    faiss_index,
    metadata_store,
    features,
    tracklets_by_camera,
    output_dir=RUN_DIR / "stage4",
)
stage4_min = (time.time() - start) / 60.0
print(f"Stage 4 complete in {stage4_min:.2f} min: {len(trajectories)} global trajectories")

start = time.time()
evaluation_result = run_stage5(
    cfg,
    trajectories,
    output_dir=RUN_DIR / "stage5",
)
stage5_min = (time.time() - start) / 60.0
print(f"Stage 5 complete in {stage5_min:.2f} min")

## 7. Save IDF1 Summary

In [ ]:
def load_metrics(report_path: Path) -> dict:
    if not report_path.exists():
        return {}
    payload = json.loads(report_path.read_text(encoding="utf-8"))
    details = payload.get("details", {}) or {}
    error_analysis = details.get("error_analysis", {}) or {}
    return {
        "mtmc_idf1": payload.get("mtmc_idf1", details.get("mtmc_idf1", payload.get("idf1"))),
        "trackeval_idf1": payload.get("idf1"),
        "mota": payload.get("mota"),
        "hota": payload.get("hota"),
        "id_switches": payload.get("id_switches"),
        "conflations": error_analysis.get("conflated_pred"),
        "fragmentations": error_analysis.get("fragmented_gt"),
        "num_pred_ids": payload.get("num_pred_ids", error_analysis.get("total_pred")),
    }

report_path = RUN_DIR / "stage5" / "evaluation_report.json"
metrics = load_metrics(report_path)
idf1_value = metrics.get("mtmc_idf1")
if idf1_value is None:
    idf1_value = metrics.get("trackeval_idf1")
if idf1_value is None:
    raise RuntimeError(f"IDF1 not found in {report_path}")

checkpoint_path = Path("/kaggle/working/checkpoint.tar.gz")
metadata_path = Path("/kaggle/working/run_metadata.json")
summary_path = Path("/kaggle/working/14a_summary.json")

summary = {
    "run_name": RUN_NAME,
    "source_10a_run_name": PREV_RUN_NAME,
    "experiment": "14a-sam2-masked-stage2",
    "kernel": "yahiaakhalafallah/14a-sam2-masked-stage2",
    "idf1": idf1_value,
    "mtmc_idf1": metrics.get("mtmc_idf1"),
    "trackeval_idf1": metrics.get("trackeval_idf1"),
    "mota": metrics.get("mota"),
    "hota": metrics.get("hota"),
    "id_switches": metrics.get("id_switches"),
    "conflations": metrics.get("conflations"),
    "fragmentations": metrics.get("fragmentations"),
    "num_pred_ids": metrics.get("num_pred_ids"),
    "sam2_model": "sam2_hiera_base_plus",
    "sam2_weights": "internet_download_meta_public_url",
    "sam2_checkpoint": str(SAM2_CKPT),
    "dilate_px": 5,
    "bbox_expand": 1.10,
    "frame_id_convention": "0-based internal Stage 1/2 frame IDs; MOT output is converted to 1-based in Stage 5",
    "stage2_outputs": {
        "features_in_memory": len(features),
        "primary_embeddings": str(PRIMARY_EMBEDDINGS_PATH),
        "tertiary_embeddings": str(TERTIARY_DINOV2_PATH),
    },
    "fusion_config": {
        "w_primary": W_PRIMARY,
        "w_tertiary": W_TERTIARY,
        "w_secondary": 0.0,
        "aqe_k": AQE_K,
        "fic_regularisation": FIC_REG,
        "pca_components": 384,
        "similarity_threshold": SIM_THRESH,
        "algorithm": ALGORITHM,
    },
    "stage_timings_min": {
        "stage2": round(elapsed / 60.0, 2),
        "stage3": round(stage3_min, 2),
        "stage4": round(stage4_min, 2),
        "stage5": round(stage5_min, 2),
    },
    "paths": {
        "run_dir": str(RUN_DIR),
        "evaluation_report": str(report_path),
        "stage3": str(RUN_DIR / "stage3"),
        "stage4": str(RUN_DIR / "stage4"),
        "stage5": str(RUN_DIR / "stage5"),
    },
}

metadata_path.write_text(json.dumps({"run_name": RUN_NAME}, indent=2), encoding="utf-8")
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")

print(f"Building checkpoint for {RUN_NAME}")
with tarfile.open(str(checkpoint_path), "w:gz") as tar:
    tar.add(str(metadata_path), arcname="run_metadata.json")
    tar.add(str(summary_path), arcname="14a_summary.json")
    for stage in ["stage1", "stage2", "stage3", "stage4", "stage5"]:
        stage_dir = RUN_DIR / stage
        count = 0
        if stage_dir.exists():
            for file_path in stage_dir.rglob("*"):
                if file_path.is_file():
                    tar.add(str(file_path), arcname=f"{RUN_NAME}/{stage}/{file_path.relative_to(stage_dir)}")
                    count += 1
        print(f"  added {stage}: {count} files")

    gt_root = EXTRACT_DIR / "gt_annotations"
    gt_count = 0
    if gt_root.exists():
        for file_path in gt_root.rglob("*"):
            if file_path.is_file():
                tar.add(str(file_path), arcname=f"gt_annotations/{file_path.relative_to(gt_root)}")
                gt_count += 1
    print(f"  added gt_annotations: {gt_count} files")

size_mb = checkpoint_path.stat().st_size / 1024**2
print(f"Saved summary: {summary_path}")
print(f"Checkpoint: {checkpoint_path} ({size_mb:.1f} MB)")
print(json.dumps(summary, indent=2))